<br>

# POST

<!--<badge>--><a href="https://colab.research.google.com/github/open-geodata/pyFBDS/blob/main/docs/scripts/post/01_post.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a><!--</badge>-->

Este notebook demonstra como obter os dados espaciais da FBDS usando o pacote `pyFBDS`. A abordagem envia uma requisição HTTP `POST` ao portal para baixar o arquivo compactado (`.tar`) de um município e, em seguida, lê um dos _layers_ como um `GeoDataFrame`, sem exigir a extração manual do arquivo.

O fluxo apresentado é:

1. definir as pastas de saída e de logs;
2. configurar o logger e instanciar o cliente `FBDS`;
3. baixar o arquivo do município e conferir os _layers_ disponíveis;
4. carregar um _layer_ e visualizar suas geometrias em um mapa interativo.

A célula de instalação abaixo é opcional quando `pyFBDS` já estiver instalado. Para executar no Google Colab, use o badge acima e ajuste o caminho do projeto na seção **Pastas**.

<br>

Michel Metran\
Data: 21.09.2026\
Atualizado em: 13.11.2026


In [ ]:
# Descomente, para instalar
#!pip3 install pyFBDS

In [ ]:
from pathlib import Path

import xyzservices.providers as xyz

from pyFBDS.logger import FBDSLogger
from pyFBDS.post import FBDS

<br>

---

## Pastas

Define os diretórios usados pelo exemplo. `data/log` recebe os registros do logger e `data/output` recebe o arquivo `.tar` baixado e o cache HTTP. As pastas são criadas caso ainda não existam.

O caminho inicial considera a execução no repositório local. No Colab, descomente a alternativa indicada na célula de código para usar o diretório de trabalho do notebook.


In [ ]:
project_path = Path(".").absolute().parents[2]

# No Colab, descomente essa
# project_path = Path('.').absolute()

# Diretório de saída
data_path = project_path / "data"

logs_path = data_path / "log"
logs_path.mkdir(parents=True, exist_ok=True)

output_path = data_path / "output"
output_path.mkdir(parents=True, exist_ok=True)
output_path

<br>

---

## FBDS

O `FBDSLogger` registra o andamento das requisições e dos downloads em `data/log`; `new_session=True` inicia uma nova sessão de log e `console=True` também mostra as mensagens durante a execução.

A classe `FBDS` usa `output_path` para salvar os dados baixados e `temp_path` para arquivos temporários e cache. Neste exemplo, os dois apontam para `data/output`, permitindo reutilizar respostas armazenadas em cache.


In [ ]:
# Instancia Logger
# Inicializa o logger uma única vez antes do loop
logger = FBDSLogger(
    log_dir=logs_path,
    new_session=True,
    console=True,
)

# Instancia FBDS
fbds = FBDS(
    temp_path=output_path,
    output_path=output_path,
    logger=logger,
)

<br>

---

## _Download_

Informe o nome do município e a sigla da Unidade da Federação conforme aparecem no portal. A chamada `download` solicita o pacote de dados do município e retorna em `filepath` o caminho do arquivo `.tar` salvo em `output_path`.


In [ ]:
# Parâmetro
municipio = "SANTOS"
uf = "SP"

In [ ]:
filepath = fbds.download(
    uf=uf,
    municipality=municipio,
    max_download_seconds=60 * 3,
)

<br>

---

## Lista Arquivos

`list_shapefiles` lista os caminhos dos arquivos `.shp` que estão dentro do `.tar`, sem extraí-los. `clean_list` simplifica esses caminhos e retorna os nomes dos _layers_ que podem ser usados na leitura. A lista pode variar conforme o município.


In [ ]:
list_shps = fbds.list_shapefiles(filepath=filepath)
list_shps

In [ ]:
list_shps = fbds.clean_list(filepath=filepath)
list_shps

<br>

---

## Read

A função `read_data` recebe a UF, o município e o nome do _layer_, e retorna os dados como um `GeoDataFrame` do GeoPandas. Ela abre o _shapefile_ diretamente dentro do `.tar`, sem extração manual. Como o arquivo já foi solicitado na seção anterior, o cache HTTP pode evitar baixá-lo novamente.

Use o nome retornado por `clean_list` para preencher `layer`. Depois da leitura, é possível filtrar, analisar ou exportar o `GeoDataFrame`. A célula final usa `gdf.explore` para criar um mapa interativo com imagens de satélite como base.


In [ ]:
# Parâmetro
layer = "MASSAS_DAGUA"
layer = "APP_USO"

# Cria geodataframe
gdf = fbds.read_data(
    uf=uf,
    municipality=municipio,
    layer=layer,
)

In [ ]:
gdf.explore(
    tiles=xyz.Esri.WorldImagery,
)